# Week 10: CNN บนภาพอาหาร — Google Colab

ฝึก MLP และ CNN บน Pizza / Steak / Sushi ด้วย PyTorch แล้ว Save/Load และทำนายภาพ
CNN ใช้ Conv สองชั้น ตามด้วย Flatten → MLP (Hidden 128 → ReLU → Output 3)

## ก่อนเริ่ม

1. เลือก **Runtime → Change runtime type → T4 GPU** (ถ้าไม่มี ใช้ CPU ได้แต่ช้ากว่า)
2. รันเซลล์ติดตั้ง Libraries แล้วดาวน์โหลด Dataset ZIP จาก GitHub ในเซลล์ถัดไป
3. รันบทเรียนจากบนลงล่าง ข้อมูลอยู่ที่ `/content/Dataset` ไม่ต้อง Mount Google Drive
4. ภาพประกอบโหลดจาก GitHub จึงต้องมีอินเทอร์เน็ต แต่ไม่ต้องอัปโหลดโฟลเดอร์ assets

ZIP เตรียมโฟลเดอร์ `Dataset/train`, `valid`, `test` ไว้แล้ว รวม 600 ภาพ
สมาชิกแต่ละชุดเหมือนฉบับเครื่อง: Train 360 / Validation 90 / Test 150 ไม่แบ่งข้อมูลใหม่
ถ้า Runtime ถูกรีเซ็ต ให้รัน Download Cell ใหม่ ไฟล์ใน `/content` ไม่ได้เก็บถาวร
ดาวน์โหลดน้ำหนักจากส่วนเสริมท้ายบทก่อนปิด Runtime


## เตรียม Colab: ติดตั้งเครื่องมือ

ใช้ PyTorch, torchvision และไลบรารีพื้นฐานที่ Colab มีอยู่แล้ว ไม่ติดตั้งทับชุด CUDA
ติดตั้งเฉพาะเครื่องมือเพิ่มเติมในเซลล์นี้ก่อน Import

In [ ]:
%pip install -q torchinfo tqdm livelossplot ipywidgets gradio

## การดาวน์โหลด Dataset จาก GitHub

ดาวน์โหลด ZIP จาก Week 10 และตรวจ SHA-256 ก่อนแตกไฟล์ ใช้ Train/Validation/Test Split เดิม
หากมี ZIP อยู่แล้วจะใช้ไฟล์เดิม การแตก ZIP ไม่เขียนทับภาพที่มีอยู่ หากต้องการเริ่มใหม่ให้ใช้ Runtime ใหม่
ข้อมูลใน `/content` ไม่ถาวร เมื่อ Runtime ถูกล้างให้ประมวลผลเซลล์นี้อีกครั้ง
แหล่งที่มาและเงื่อนไขการใช้ภาพอยู่ใน `Dataset/README.md`

In [ ]:
%cd /content
import hashlib
from pathlib import Path
from urllib.request import urlretrieve

DATASET_URL = 'https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W10_CNN_TransferLearning/food_pizza_steak_sushi_colab.zip'
DATASET_ZIP = Path('/content/food_pizza_steak_sushi_colab.zip')
EXPECTED_SHA256 = '8da3c2870ddff28cafcca222939bdbb1f909506c6e8f6833386e01eaef53b431'

if not DATASET_ZIP.is_file():
    urlretrieve(DATASET_URL, DATASET_ZIP)  # ดาวน์โหลด ZIP โดยไม่ต้อง Upload จากเครื่อง

actual_sha256 = hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError('Dataset ZIP: SHA-256 mismatch; ตรวจหรือลบ ZIP ที่ไม่สมบูรณ์แล้วดาวน์โหลดใหม่')

!unzip -q -n /content/food_pizza_steak_sushi_colab.zip -d /content
print('Dataset พร้อมใช้:', Path('/content/Dataset'))

## 0. Load Libraries


In [ ]:
import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchinfo import summary
from tqdm.auto import tqdm
from livelossplot import PlotLosses
from IPython.display import display
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

### เตรียมอุปกรณ์และ Seed


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print('Device:', DEVICE)

## 1. อ่านภาพด้วย ImageFolder

`ImageFolder` อ่านภาพ RGB และใช้ชื่อโฟลเดอร์เป็น Label


In [ ]:
DATA_DIR = Path('Dataset')
raw_dataset = datasets.ImageFolder(DATA_DIR / 'train')
CLASS_NAMES = raw_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print('Classes:', raw_dataset.class_to_idx)
print('Train images:', len(raw_dataset))

### เปิดภาพตัวอย่าง

เลือกภาพจาก Train แล้วดู Label ขนาดภาพ และจำนวน Channel


In [ ]:
raw_image, raw_label = raw_dataset[0]
print('Class:', CLASS_NAMES[raw_label])
print('PIL:', raw_image.mode, raw_image.size)
print('Array [H, W, C]:', np.asarray(raw_image).shape)
plt.imshow(raw_image)
plt.title(CLASS_NAMES[raw_label])
plt.axis('off')
plt.show()

## 2. เตรียมภาพก่อนเข้าโมเดล

Resize เป็นสี่เหลี่ยม จากนั้นแปลงเป็น Tensor และ Normalize
กำหนด STD และ MEAN เป็น 0.5 เพื่อให้ค่าพิกเซลอยู่ระหว่าง -1 ถึง 1 (ถ้างานจริง ควรใช้ค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของ Dataset จริง)


In [ ]:
IMAGE_SIZE = 64
MEAN = (0.5, 0.5, 0.5)
STD = (0.5, 0.5, 0.5)

def show_images(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3), squeeze=False)
    for index, ax in enumerate(axes[0]):
        ax.imshow(images[index])
        ax.set_title(titles[index])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def image_for_display(tensor, mean=MEAN, std=STD):
    image = tensor.detach().cpu()
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    image = image * std + mean            # ย้อน Normalize แยกแต่ละ Channel
    image = image.clamp(0, 1)             # ช่วงสีสำหรับแสดงผล
    image = image.permute(1, 2, 0)        # [C, H, W] -> [H, W, C]
    return image.numpy()

example_image, example_label = raw_image, raw_label
image_tensor = eval_transform(example_image)
print('Tensor [C, H, W]:', image_tensor.shape, image_tensor.dtype)
print('Batch [N, C, H, W]:', image_tensor.unsqueeze(0).shape)
print('Normalized range:', image_tensor.min().item(), image_tensor.max().item())
show_images([example_image, image_for_display(image_tensor)],
            ['Original RGB', 'Actual input (display scale)'])

### Augmentation เฉพาะ Train

สุ่มพลิก หมุน และปรับสี ตรวจว่าภาพยังเป็นคลาสเดิม มุมดำเกิดจากการหมุน ส่วน Validation/Test ห้ามทำการ Augmentation

การ Augmentation ช่วยให้โมเดลเรียนรู้ได้ดีขึ้น และเป็นอย่างแรกที่ควรทำเพื่อให้ Model สามารถนำไป Run ในงานจริงได้


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

torch.manual_seed(42)
augmented_images = [example_image]
augmented_titles = ['Original']
for i in range(1, 5):
    changed_image = train_transform(example_image)
    augmented_images.append(image_for_display(changed_image))
    augmented_titles.append(f'Train view {i}')
show_images(augmented_images, augmented_titles)

### อ่านแต่ละโฟลเดอร์และสร้าง DataLoader

เตรียม Train 360 / Validation 90 / Test 150 ไว้แล้ว แต่ละภาพอยู่ชุดเดียว ใช้ Augmentation เฉพาะ Train และเริ่มด้วย `num_workers=0`


In [ ]:
BATCH_SIZE = 32
train_set = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)

# ใช้ Eval Transform สำหรับ Validation และ Test Set เพื่อให้โมเดลเห็นข้อมูลที่ไม่ผ่าน Augmentation
val_set = datasets.ImageFolder(DATA_DIR / 'valid', transform=eval_transform)

test_set = datasets.ImageFolder(DATA_DIR / 'test', transform=eval_transform)


train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Train / Validation / Test:', len(train_set), len(val_set), len(test_set))
images, labels = next(iter(train_loader))
print('Images:', images.shape, images.dtype)
print('Labels:', labels.shape, labels.dtype)

## 3. MLP บนภาพ RGB

Flatten ภาพ 3×64×64 เป็น 12,288 ค่า ส่ง Logits เข้า `CrossEntropyLoss`


In [ ]:
class ImageMLP(nn.Module):
    def __init__(self, num_classes, image_size=64):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * image_size * image_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.fc2(x)
        return logits

torch.manual_seed(42)
# ดู Shape/Parameters
mlp = ImageMLP(NUM_CLASSES, IMAGE_SIZE)
summary(mlp, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device='cpu',
        col_names=['input_size', 'output_size', 'num_params'])

## 4. Train MLP พร้อม Validation

ฝึก 10 Epochs ดูกราฟ Train/Validation แล้วโหลดน้ำหนักจาก Epoch ที่ Validation Loss ต่ำสุด


In [ ]:
EPOCHS = 10  # ใช้จำนวน Epochs เท่ากันทั้งสองโมเดล
torch.manual_seed(42)
mlp = ImageMLP(NUM_CLASSES, IMAGE_SIZE).to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')

for epoch in range(EPOCHS):
    # Train
    mlp.train()
    total_loss, total_correct = 0.0, 0
    for batch_images, batch_labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        batch_images = batch_images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient จาก Batch ก่อน เพราะ PyTorch สะสม Gradient ไว้
        logits = mlp(batch_images)  # Forward: คำนวณคะแนนของแต่ละคลาส ยังไม่ใช่ความน่าจะเป็น
        loss = loss_fn(logits, batch_labels)  # เทียบคะแนนกับ Label จริง ได้ Loss เฉลี่ยของ Batch
        loss.backward()  # คำนวณ Gradient ว่าน้ำหนักแต่ละตัวส่งผลต่อ Loss อย่างไร
        optimizer.step()  # ใช้ Gradient ปรับน้ำหนักด้วย Adam
        # item() แปลง Tensor ค่าเดียวเป็นตัวเลข; คูณจำนวนภาพเพื่อสะสม Loss รวมก่อนเฉลี่ยทั้ง Epoch
        total_loss += loss.item() * len(batch_labels)
        # argmax(dim=1) เลือกคลาสคะแนนสูงสุดของแต่ละภาพ แล้วเทียบกับ Label จริง
        # sum() นับภาพที่ทายถูกใน Batch; item() แปลงเป็นตัวเลข แล้วสะสมไว้หา Accuracy ของ Epoch
        total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
    train_loss = total_loss / len(train_set)
    train_acc = total_correct / len(train_set)

    # Validation: ไม่ปรับน้ำหนัก
    mlp.eval()
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():
        for batch_images, batch_labels in val_loader:
            batch_images = batch_images.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)
            logits = mlp(batch_images)
            loss = loss_fn(logits, batch_labels)
            total_loss += loss.item() * len(batch_labels)
            total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
    val_loss = total_loss / len(val_set)
    val_acc = total_correct / len(val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()

    if val_loss < best_loss:
        best_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(mlp.state_dict())

mlp.load_state_dict(best_state)
mlp.eval()
mlp_result = {'history': history, 'best_epoch': best_epoch}
print(f'MLP: Best epoch {best_epoch}, Validation loss={best_loss:.4f}')

## 5. CNN

ภาพนี้วาดให้ตรงกับ `SimpleCNN` ด้านล่าง อ่านจากซ้ายไปขวา: ภาพอาหารจริง → Conv/ReLU/Pool สองรอบ → Flatten → MLP (Hidden Layer → ReLU → Output) ให้คะแนน 3 คลาส

![โครงสร้าง SimpleCNN พร้อมภาพอาหารจริง หน้าที่ของแต่ละ Layer และขนาดข้อมูล](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W10_CNN_TransferLearning/assets/food_cnn_architecture.png)

[เปิดภาพขนาดเต็ม](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W10_CNN_TransferLearning/assets/food_cnn_architecture.png) · [ภาพเวกเตอร์ SVG](https://github.com/vasuprint/AI-CE564/blob/main/weeks/W10_CNN_TransferLearning/assets/food_cnn_architecture.svg)

**แต่ละชั้นทำอะไร** (ขนาดเขียนเป็น Channels × Height × Width ไม่รวม Batch):

| ชั้น                  | หน้าที่                                                                                                                     | ขนาดหลังผ่านชั้น |
| --------------------- | --------------------------------------------------------------------------------------------------------------------------- | ---------------- |
| Input                 | ภาพ RGB ผ่าน Resize เป็น 64×64 และ Normalize ตามที่เตรียมไว้                                                                | 3 × 64 × 64      |
| Conv1                 | ใช้ Filter ที่เรียนรู้ได้ 16 ตัว กวาดบริเวณย่อยของ Input โดยแต่ละ Filter รับครบ 3 Channels; Kernel 3×3, Stride 1, Padding 1 | 16 × 64 × 64     |
| ReLU                  | เปลี่ยนค่าลบเป็น 0 เก็บค่าบวกเดิม เพิ่มความไม่เป็นเชิงเส้น ขนาดไม่เปลี่ยน                                                   | 16 × 64 × 64     |
| MaxPool               | เลือกค่าสูงสุดในแต่ละบริเวณ 2×2, Stride 2 ลดความสูง/กว้างครึ่งหนึ่ง ไม่ลด Channels                                          | 16 × 32 × 32     |
| Conv2                 | ใช้ Filter ที่เรียนรู้ได้ 32 ตัว แต่ละตัวรับครบ 16 Channels จากชั้นก่อน; Kernel 3×3, Stride 1, Padding 1                    | 32 × 32 × 32     |
| ReLU                  | เปลี่ยนค่าลบเป็น 0 เช่นเดิม ขนาดไม่เปลี่ยน                                                                                  | 32 × 32 × 32     |
| MaxPool               | เลือกค่าสูงสุดในบริเวณ 2×2 อีกครั้ง                                                                                         | 32 × 16 × 16     |
| Flatten               | จัดเรียงค่าทั้งหมดเป็นเวกเตอร์ ไม่ได้เรียนรู้น้ำหนักหรือลดจำนวนค่า                                                          | 8192 ค่า         |
| Hidden Layer (Linear) | เรียนรู้น้ำหนักเพื่อรวม 8192 Features เป็นค่าซ่อน 128 ค่า                                                                   | 128 ค่า          |
| ReLU                  | เปลี่ยนค่าลบเป็น 0 ทำให้ MLP เรียนรู้ความสัมพันธ์แบบไม่เป็นเชิงเส้นได้                                                      | 128 ค่า          |
| Output Layer (Linear) | รวมค่าจาก Hidden Layer เป็นคะแนนดิบของ Pizza / Steak / Sushi ไม่ใส่ ReLU หลังชั้นนี้                                        | 3 Logits         |

ส่วนจำแนก `classifier` คือ **MLP: Linear(8192, 128) → ReLU → Linear(128, 3)** ส่วน Flatten มีหน้าที่จัดรูปข้อมูลก่อนเข้า MLP; 128 คือจำนวน Hidden Units ที่เรากำหนด ไม่ใช่จำนวนคลาส

ภาพ Input มาจาก `Dataset/train/pizza/1001116.jpg` ส่วนลวดลาย Feature Maps และจำนวนแผ่นที่วาดเป็นภาพสื่อแนวคิด **ไม่ใช่ Activations จากโมเดลที่ฝึกแล้ว** ดูแหล่งภาพใน [assets/README.md](https://github.com/vasuprint/AI-CE564/blob/main/weeks/W10_CNN_TransferLearning/assets/README.md)

ใช้ `torchinfo` ตรวจโมเดลจริงได้ ไม่ต้องรัน Demo ของแต่ละ Layer แยกอีก ส่ง Logits เข้า `CrossEntropyLoss` โดยตรง; ใช้ `argmax` เลือกคลาส และใช้ Softmax เฉพาะเมื่อต้องการแสดงความน่าจะเป็น


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, image_size=64):
        super().__init__()
        if image_size < 4:
            raise ValueError('image_size ต้องไม่น้อยกว่า 4')
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # RGB 3 channels → 16 filters
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # รับ 16 channels → 32 filters
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        feature_size = image_size // 2 // 2  # Pool สองครั้ง: 64 → 32 → 16
        flattened_size = 32 * feature_size * feature_size  # Channels × Height × Width = 8192
        self.classifier = nn.Sequential(  # MLP สำหรับจำแนก Features จาก CNN
            nn.Linear(flattened_size, 128),  # Hidden Layer: 8192 → 128 เมื่อภาพ 64×64
            nn.ReLU(),                      # Activation ของ Hidden Layer
            nn.Linear(128, num_classes),     # Output: 128 → 3 Logits ไม่ใส่ Softmax
        )

    def forward_features(self, x):
        x = self.conv1(x)  # [N, 16, 64, 64] เมื่อ size=64
        x = self.relu(x)
        x = self.pool(x)   # [N, 16, 32, 32]
        x = self.conv2(x)  # [N, 32, 32, 32]
        x = self.relu(x)
        x = self.pool(x)   # [N, 32, 16, 16]
        features = self.flatten(x)
        return features

    def forward(self, x):
        features = self.forward_features(x)
        logits = self.classifier(features)
        return logits

torch.manual_seed(42)
# ใช้ดู Shape/Parameters; เซลล์ Train จะสร้างใหม่ด้วย Seed เดิม
cnn = SimpleCNN(NUM_CLASSES, IMAGE_SIZE)
summary(cnn, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device='cpu',
        col_names=['input_size', 'output_size', 'num_params'])
# TODO: คำนวณจำนวน Parameters ของ Conv1 แล้วเทียบกับ torchinfo
# ถ้าแก้ 16/32 ต้องแก้ Input ของ Conv2 และ Linear ให้ตรงกัน แล้ว Restart Kernel และ Run All

### Train CNN พร้อม Validation

ใช้ขั้นตอนเดียวกับ MLP แล้วเปรียบเทียบกราฟ Train/Validation และ Best Epoch

หลังเปลี่ยนส่วนจำแนกเป็น MLP ต้องรันเซลล์ CNN และ Train CNN ใหม่ แล้วรันส่วนประเมิน/Save/Load ต่อ กราฟหรือผล CNN ที่ยังค้างจากการรันเก่าไม่ใช่ผลของโครงสร้างใหม่ และน้ำหนักจาก Linear ชั้นเดียวใช้กับโมเดลใหม่นี้ไม่ได้


In [ ]:
torch.manual_seed(42)
cnn = SimpleCNN(NUM_CLASSES, IMAGE_SIZE).to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')

for epoch in range(EPOCHS):
    # Train
    cnn.train()
    total_loss, total_correct = 0.0, 0
    for batch_images, batch_labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        batch_images = batch_images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient จาก Batch ก่อน เพราะ PyTorch สะสม Gradient ไว้
        logits = cnn(batch_images)  # Forward: คำนวณคะแนนของแต่ละคลาส ยังไม่ใช่ความน่าจะเป็น
        loss = loss_fn(logits, batch_labels)  # เทียบคะแนนกับ Label จริง ได้ Loss เฉลี่ยของ Batch
        loss.backward()  # คำนวณ Gradient ว่าน้ำหนักแต่ละตัวส่งผลต่อ Loss อย่างไร
        optimizer.step()  # ใช้ Gradient ปรับน้ำหนักด้วย Adam
        # item() แปลง Tensor ค่าเดียวเป็นตัวเลข; คูณจำนวนภาพเพื่อสะสม Loss รวมก่อนเฉลี่ยทั้ง Epoch
        total_loss += loss.item() * len(batch_labels)
        # argmax(dim=1) เลือกคลาสคะแนนสูงสุดของแต่ละภาพ แล้วเทียบกับ Label จริง
        # sum() นับภาพที่ทายถูกใน Batch; item() แปลงเป็นตัวเลข แล้วสะสมไว้หา Accuracy ของ Epoch
        total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
    train_loss = total_loss / len(train_set)
    train_acc = total_correct / len(train_set)

    # Validation: ไม่ปรับน้ำหนัก
    cnn.eval()
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():
        for batch_images, batch_labels in val_loader:
            batch_images = batch_images.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)
            logits = cnn(batch_images)
            loss = loss_fn(logits, batch_labels)
            total_loss += loss.item() * len(batch_labels)
            total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
    val_loss = total_loss / len(val_set)
    val_acc = total_correct / len(val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()

    if val_loss < best_loss:
        best_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(cnn.state_dict())

cnn.load_state_dict(best_state)
cnn.eval()
cnn_result = {'history': history, 'best_epoch': best_epoch}
print(f'CNN: Best epoch {best_epoch}, Validation loss={best_loss:.4f}')

## 6. Validation

ใช้ผลจาก Best Epoch ใน History เปรียบเทียบกับ MLP ที่เรียนมาแล้ว จากนี้ใช้ CNN ต่อโดยไม่เพิ่มขั้นตอนเลือกสลับโมเดล


In [ ]:
results = {'MLP': mlp_result, 'CNN': cnn_result}
rows = []
for name, result in results.items():
    index = result['best_epoch'] - 1
    rows.append({'Model': name, 'Best epoch': result['best_epoch'],
                 'Val loss': result['history']['val_loss'][index],
                 'Val accuracy': result['history']['val_accuracy'][index]})
comparison_table = pd.DataFrame(rows)
display(comparison_table.round(4))
print('Validation ทายถูกเพิ่ม 1 ภาพ เปลี่ยน Accuracy:', 1 / len(val_set))

### ดูภาพที่ CNN ทำนายผิด

ใช้ Validation ระหว่างพัฒนา ไม่ใช้ Test ปรับโมเดล


In [ ]:
cnn.eval()
val_true, val_pred = [], []
with torch.no_grad():
    for batch_images, batch_labels in val_loader:
        logits = cnn(batch_images.to(DEVICE))
        val_true.extend(batch_labels.tolist())
        val_pred.extend(logits.argmax(dim=1).cpu().tolist())
val_true = np.array(val_true)
val_pred = np.array(val_pred)
wrong = np.flatnonzero(val_pred != val_true)[:6]
if len(wrong):
    wrong_images = []
    wrong_titles = []
    for index in wrong:
        image, label = val_set[int(index)]
        predicted_label = val_pred[index]
        wrong_images.append(image_for_display(image))
        wrong_titles.append(f'True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[predicted_label]}')
    show_images(wrong_images, wrong_titles)
else:
    print('ไม่พบภาพทายผิดใน Validation รอบนี้ ไม่ได้แปลว่าจะถูกทุกภาพใหม่')

### ประเมิน CNN บน Test ตอนจบ

รันเมื่อปรับ CNN ด้วย Validation เสร็จแล้ว ห้ามใช้ผล Test กลับไปปรับโมเดล หาก Run All จะประเมิน Test ด้วย

PyTorch ไม่มี `model.evaluate()` แบบ Keras; `eval()` เปลี่ยนโหมดเท่านั้น ส่วน Loop นี้คำนวณ Loss/Accuracy และไม่ปรับน้ำหนัก


In [ ]:
cnn.eval()
total_loss, total_correct = 0.0, 0
test_true, test_pred = [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        logits = cnn(batch_images)
        loss = loss_fn(logits, batch_labels)
        predictions = logits.argmax(dim=1)
        total_loss += loss.item() * len(batch_labels)
        total_correct += (predictions == batch_labels).sum().item()
        test_true.extend(batch_labels.cpu().tolist())
        test_pred.extend(predictions.cpu().tolist())
test_loss = total_loss / len(test_set)
test_acc = total_correct / len(test_set)
test_metrics = {'model': 'CNN', 'loss': test_loss, 'accuracy': test_acc}
print(f'CNN: Test loss={test_loss:.4f}, accuracy={test_acc:.1%}')
print(classification_report(test_true, test_pred, labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(test_true, test_pred, labels=list(range(NUM_CLASSES)),
                                         display_labels=CLASS_NAMES, cmap='Blues')
plt.title('CNN: held-out test')
plt.tight_layout()
plt.show()

## 7. ดู Feature Maps

เปรียบเทียบ Conv1 และ Conv2 ของ CNN ที่ฝึกแล้ว โดยใช้ภาพเดียวกัน โมเดลนี้มี Conv สองชั้น **ยังไม่มี Conv3**

- **Conv1 + ReLU:** โดยทั่วไปเป็น Low-level Features เช่น ขอบ การเปลี่ยนสี และลวดลายพื้นฐาน
- **Conv2 + ReLU:** รวม Features จากชั้นก่อนและมองบริเวณ Input กว้างขึ้น จึงอาจเป็นลวดลายหรือส่วนประกอบที่ซับซ้อนกว่า (Higher-level เมื่อเทียบกับ Conv1)
- ยิ่งลึก โดยทั่วไปยิ่งมีโอกาสเรียนรู้ Features ระดับสูงขึ้น แต่ CNN สองชั้นนี้ยังไม่จำเป็นต้องมี Channel ที่ตรวจจับ “พิซซ่า” หรือ “ซูชิ” โดยตรง ต้องทดลองหลายภาพ ไม่สรุปจากภาพเดียว

แถวบนแสดง 4 Channels แรกของ Conv1 แถวล่างของ Conv2; หมายเลข Channel เดียวกันคนละชั้นไม่ได้แทน Feature เดียวกัน ภาพสีเทาปรับสเกลแยกต่อ Channel: สว่างคือ Activation สูงภายในภาพนั้น จึงเทียบความสว่างข้ามภาพไม่ได้ และ Feature Maps ไม่ใช่แผนที่ความสำคัญต่อคำตอบของโมเดล


In [ ]:
cnn.eval()
feature_image, feature_label = val_set[0]
feature_batch = feature_image.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    conv1_maps = cnn.relu(cnn.conv1(feature_batch))  # หลัง Conv1 + ReLU ก่อน Pool1
    pooled_maps = cnn.pool(conv1_maps)             # Conv2 รับผลหลัง Pool1 ไม่ใช่ภาพ RGB
    conv2_maps = cnn.relu(cnn.conv2(pooled_maps))   # หลัง Conv2 + ReLU ก่อน Pool2
    conv1_maps = conv1_maps.cpu()
    conv2_maps = conv2_maps.cpu()
plt.imshow(image_for_display(feature_image))
plt.title(f'Feature input: {CLASS_NAMES[feature_label]}')
plt.axis('off')
plt.show()
n_maps = min(4, conv1_maps.shape[1], conv2_maps.shape[1])
fig, axes = plt.subplots(2, n_maps, figsize=(3 * n_maps, 6), squeeze=False)
for row, (layer_name, maps) in enumerate([('Conv1 + ReLU', conv1_maps),
                                         ('Conv2 + ReLU', conv2_maps)]):
    for channel in range(n_maps):
        ax = axes[row, channel]
        ax.imshow(maps[0, channel].numpy(), cmap='gray', interpolation='nearest')
        ax.set_title(f'{layer_name}: channel {channel}\n{maps.shape[2]} × {maps.shape[3]}')
        ax.axis('off')
plt.tight_layout()
plt.show()

### Feature Vector และ Logits

`forward_features()` Return ค่าก่อน Layer Output สำหรับภาพ 64×64 ได้ 32 × 16 × 16 = 8192 ค่า


In [ ]:
with torch.no_grad():
    feature_vectors = cnn.forward_features(feature_batch)
    feature_logits = cnn.classifier(feature_vectors)
print('Input:', feature_batch.shape)
print('Feature vector:', feature_vectors.shape)
print('Logits:', feature_logits.shape)
print('First 12 feature values:', feature_vectors[0, :12])

## 8. Save CNN

บันทึกเฉพาะน้ำหนักจาก Best Epoch ด้วย `state_dict()`

ไฟล์ weight ไม่ได้เก็บโค้ดโมเดลหรือ Preprocessing เมื่อใช้ภายหลังต้องมี `SimpleCNN`, `IMAGE_SIZE`, `CLASS_NAMES` ตามลำดับเดิม และ `eval_transform` เหมือนตอนฝึก


In [ ]:
OUT_DIR = Path('cnn_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / 'food_cnn_weights.pt'
torch.save(cnn.state_dict(), MODEL_PATH)  # บันทึกน้ำหนัก ไม่บันทึกโมเดลทั้งออบเจ็กต์
print('Saved:', MODEL_PATH)

### Load CNN

สร้างโครงสร้างเดิม → อ่านน้ำหนัก → ใส่น้ำหนัก → ตั้งโหมดทำนาย ใช้ CPU


In [ ]:
loaded_cnn = SimpleCNN(NUM_CLASSES, IMAGE_SIZE)
weights = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
loaded_cnn.load_state_dict(weights)
loaded_cnn.eval()  # โหมดทำนาย ไม่ใช่คำสั่งคำนวณ Accuracy

### ใช้ CNN ที่โหลดมาทำนายภาพ

เริ่มจากภาพ Validation ที่รู้คำตอบ แล้วเปลี่ยน `IMAGE_PATH` เป็นภาพ JPEG/PNG ของตนเอง เช่น `Path("my_pizza.jpg")` ถ้าเป็น HEIC ให้แปลงก่อน

ใช้ `eval_transform` เดิมและเพิ่มมิติ Batch แม้จะใช้กับภาพเดียวก็ตาม


In [ ]:
IMAGE_PATH = Path(val_set.samples[0][0])  # เปลี่ยนเป็น Path('my_pizza.jpg') เพื่อลองภาพใหม่
with Image.open(IMAGE_PATH) as source_image:
    image = ImageOps.exif_transpose(source_image).convert('RGB')

image_tensor = eval_transform(image)  # เตรียมภาพเหมือน Validation ตอนฝึก
input_batch = image_tensor.unsqueeze(0)  # [3, H, W] → [1, 3, H, W]
with torch.no_grad():  # ทำนายอย่างเดียว ไม่คำนวณ Gradient
    logits = loaded_cnn(input_batch)
    probabilities = logits.softmax(dim=1)[0]

predicted_class = probabilities.argmax().item()
print('Prediction:', CLASS_NAMES[predicted_class])
for name, probability in zip(CLASS_NAMES, probabilities):
    print(f'{name}: {probability.item():.1%}')
plt.imshow(image)
plt.title(CLASS_NAMES[predicted_class])
plt.axis('off')
plt.show()

### บันทึกผล

เก็บ Metrics และค่าที่ต้องใช้คู่กับไฟล์น้ำหนัก รันซ้ำจะเขียนทับไฟล์ชื่อเดิม


In [ ]:
comparison_table.to_json(OUT_DIR / 'validation_comparison.json', orient='records', indent=2)
(OUT_DIR / 'histories.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
(OUT_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')
inference_config = {'class_names': CLASS_NAMES, 'image_size': IMAGE_SIZE,
                    'mean': list(MEAN), 'std': list(STD)}
(OUT_DIR / 'inference_config.json').write_text(json.dumps(inference_config, indent=2), encoding='utf-8')
print('Saved metrics and inference settings to:', OUT_DIR)

### PCA ของ Validation Features

ลด Features เหลือ 2 มิติเพื่อดูการกระจาย ภาพนี้ยังบอกความแม่นยำไม่ได้ และไม่ใช้เลือกโมเดลบน Test


In [ ]:
RUN_PCA = True
if RUN_PCA:
    from sklearn.decomposition import PCA

    cnn.eval()
    all_features, all_labels = [], []
    with torch.no_grad():
        for pca_images, pca_batch_labels in val_loader:
            all_features.append(cnn.forward_features(pca_images.to(DEVICE)).cpu())
            all_labels.append(pca_batch_labels)
    pca_features = torch.cat(all_features).numpy()
    pca_labels = torch.cat(all_labels).numpy()
    embedding = PCA(n_components=2, random_state=42).fit_transform(pca_features)
    for label, name in enumerate(CLASS_NAMES):
        chosen = pca_labels == label
        plt.scatter(embedding[chosen, 0], embedding[chosen, 1], label=name, alpha=0.7)
    plt.legend()
    plt.title('CNN validation features: PCA')
    plt.show()
else:
    print('ข้าม PCA; ตั้ง RUN_PCA=True เพื่อทดลอง')

### Gradio Demo บน Colab (ส่วนเสริม)

รัน Load CNN ก่อน แล้วตั้ง `RUN_GRADIO = True` เมื่อต้องการทดลองเท่านั้น
Colab ต้องใช้ `share=True` เพื่อเปิดผ่านเบราว์เซอร์ ซึ่งสร้าง **ลิงก์สาธารณะชั่วคราว**:
ใครมีลิงก์สามารถเข้า Demo ได้ อย่าใช้ภาพหรือข้อมูลลับ และอย่าเผยแพร่ลิงก์โดยไม่จำเป็น
กด Stop เซลล์เมื่อเลิกใช้; ใช้ `demo.close()` เพื่อปิด Demo ใน Runtime
ค่าเริ่มต้นปิดไว้เพื่อให้ Run All ไม่เปิดเว็บโดยอัตโนมัติ

In [ ]:
def web_predict(image):
    if image is None:
        return {}
    image = ImageOps.exif_transpose(image).convert('RGB')
    batch = eval_transform(image).unsqueeze(0)
    with torch.no_grad():
        probabilities = loaded_cnn(batch).softmax(dim=1)[0]
    return {name: probability.item() for name, probability in zip(CLASS_NAMES, probabilities)}

RUN_GRADIO = False
if RUN_GRADIO:
    import gradio as gr

    demo = gr.Interface(
        fn=web_predict,
        inputs=gr.Image(type='pil'),
        outputs=gr.Label(num_top_classes=3),
        title='Pizza / Steak / Sushi',
        description='Classroom demo: three known food classes, not a general food detector.',
    )
    demo.launch(share=True, debug=True)
else:
    print('ข้ามเว็บ Demo; เซลล์ทำนายภาพใช้ได้โดยไม่ต้องติดตั้ง Gradio')

## ดาวน์โหลดน้ำหนักกลับเครื่อง (ส่วนเสริม)

รัน Save CNN และบันทึกผลก่อน แล้วรันเซลล์นี้เมื่อต้องการดาวน์โหลด ZIP ผลลัพธ์
ไฟล์น้ำหนักต้องใช้คู่กับ `SimpleCNN` และ Preprocessing/ลำดับคลาสเดิม
เซลล์นี้ไม่รวมรูปข้อมูล Train/Validation/Test

In [ ]:
DOWNLOAD_RESULTS = False  # เปลี่ยนเป็น True เมื่อต้องการเก็บผลกลับเครื่อง
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    results_zip = shutil.make_archive('/content/food_cnn_results', 'zip', OUT_DIR)
    files.download(results_zip)